| Metric Name                         | Formula / Implementation                                                                    | Pillar Category                | Implementation Cost                  | Information Value                         | Key Reason / Integration                                                                                                                                                                                                                                                                    |
| ----------------------------------- | ------------------------------------------------------------------------------------------- | ------------------------------ | ------------------------------------ | ----------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Return Autocorrelation Lag-1**    | `Corr(returns[1:], returns[:-1])` over 15d rolling window.<br>Buffer: 15 floats.            | **Regime Detector**            | **Easy**<br>O(n), n=15               | **Critical**                              | **Prevents regime misallocation.** Detects serial correlation structure (trend vs. mean-reversion). Enables dynamic weighting: if ρ > 0, trust Momentum (Metric 6); if ρ < 0, trust Dip Buyer (Metric 4). Solves the "fighting pillars" problem.                                            |
| **Range Position (20d)**            | `(Close - Min_Low_20d) / (Max_High_20d - Min_Low_20d)`<br>Buffer: Circular min/max tracker. | **Extremes / Valuation**       | **Easy**<br>O(1) update              | **High**                                  | **State-dependent mean-reversion signal.** Unlike drawdown (path-dependent), this measures "cheap vs expensive" within recent bounds. Values <0.1 validate Dip Buyer (Metric 4); >0.9 invalidate Momentum (Metric 6). Adds non-linear price bounds.                                         |
| **OBV Divergence (5d)**             | `Slope(OBV_5d) - Slope(Close_5d)`<br>State: Running OBV total + 5d buffer.                  | **Smart Money / Confirmation** | **Easy**<br>O(1) update              | **High**<br>(Conditional on clean volume) | **Volume-price validation.** Positive divergence (OBV rising faster than price) confirms accumulation—validates Dip Buyer entries. Negative divergence (price rising, OBV flat) flags weak trend—invalidates Momentum. Only flow-based metric; orthogonal to all six price-derived metrics. |
| **Convexity (Momentum Exhaustion)** | `Mom_5d - Mom_10d`<br>(Second derivative of price)                                          | **Exit Timing**                | **Trivial**<br>Reuses existing calcs | **Medium-High**                           | **Captures acceleration decay.** Identifies when trend is slowing (bullish/bearish exhaustion). Provides exit signals missing from trend-only metrics. Orthogonal to momentum itself (measures rate of change, not direction). 


Implementation Priority Order (SD/SQ Consensus):  
- Autocorrelation → First priority; enables regime-aware switching
- Range Position → Second priority; adds mean-reversion bounds
- OBV Divergence → Third priority; confirms/denies price action (feature flag gated)
- Convexity → Fourth priority; fine-tunes exit timing using existing data  

Total Cost: All four are O(1) or O(n<20) with minimal state overhead (<100 floats total). No matrix operations. Transforms registry from "correlated trend/vol metrics" into a regime-aware, volume-confirmed trading system.

| Pillar | Winner | Reason |
| :--- | :--- | :--- |
| **1. Momentum** | `VIX Filtered Momentum` | Includes the VIX "Off-Switch" logic. |
| **2. Risk-Adjusted** | `Sharpe (ATRP)` | ATR-based volatility adjustment is more robust for ranking than standard StdDev. |
| **3. Quality** | `Consistency (WinRate 5d)` | Low overlap with Sharpe (18%), measures "steadiness" rather than just total return. |
| **4. Mean Reversion**| `Oversold (-RSI)` | Pure anti-momentum play. 0% overlap with Sharpe. |
| **5. Protection** | `Dip Buyer (-dd_21)` | Captures "Recovery Alpha." Only 29% overlap with RSI. |
| **6. Volatility** | `Low Volatility (-ATRP)` | Picks the "Quiet" stocks. 0% overlap with Momentum. |

Replace them with standard text or safe ASCII symbols:
*   `🚀` $\rightarrow$ `STARTING`
*   `🏗️` $\rightarrow$ `BUILDING`
*   `💓` $\rightarrow$ `HEARTBEAT`
*   `💾` $\rightarrow$ `SAVING`
*   `✨` $\rightarrow$ `COMPLETE`
*   `✅` $\rightarrow$ `SUCCESS`